## 1. Problem Statement

This project analyzes student-performance data to identify factors
associated with academic results.

The analysis focuses on study time, attendance, major, assignment
scores, midterm scores, and final scores.

The main questions are:

- What is the overall student performance?
- Which major has the highest average score?
- Is study time associated with academic performance?
- Is attendance associated with academic performance?
- Which students appear to be outliers?

The dataset may not include all factors affecting student performance,
such as previous academic ability, health, sleep, family conditions,
or course difficulty.

## 2. Imports and Project Paths

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [3]:
def find_project_root(start_path: Path) -> Path:
    """Find the project directory containing data, src, and notebooks."""

    start_path = start_path.resolve()

    candidates = [start_path, *start_path.parents]

    for candidate in candidates:
        has_data = (candidate / "data").exists()
        has_src = (candidate / "src").exists()
        has_notebooks = (candidate / "notebooks").exists()

        if has_data and has_src and has_notebooks:
            return candidate

    raise FileNotFoundError(
        "Could not find the project root. "
        "Make sure the project contains data/, src/, and notebooks/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)

Current working directory: c:\Users\LOQ\OneDrive\Documents\uni\machine_learning\python-for-ml\mini_projects\student_performance\notebooks
Project root: C:\Users\LOQ\OneDrive\Documents\uni\MACHINE_LEARNING\python-for-ml\mini_projects\student_performance


In [4]:
RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "student_performance.csv"
)

PROCESSED_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "student_performance_cleaned.csv"
)

FIGURES_DIR = (
    PROJECT_ROOT
    / "reports"
    / "figures"
)

SRC_DIR = PROJECT_ROOT / "src"

In [5]:
PROCESSED_DATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [6]:
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

In [7]:
print("Project root:")
print(PROJECT_ROOT)

print("\nRaw data path:")
print(RAW_DATA_PATH)

print("\nRaw data exists:")
print(RAW_DATA_PATH.exists())

print("\nProcessed data directory:")
print(PROCESSED_DATA_PATH.parent)

print("\nFigures directory:")
print(FIGURES_DIR)

print("\nSource directory:")
print(SRC_DIR)

Project root:
C:\Users\LOQ\OneDrive\Documents\uni\MACHINE_LEARNING\python-for-ml\mini_projects\student_performance

Raw data path:
C:\Users\LOQ\OneDrive\Documents\uni\MACHINE_LEARNING\python-for-ml\mini_projects\student_performance\data\raw\student_performance.csv

Raw data exists:
True

Processed data directory:
C:\Users\LOQ\OneDrive\Documents\uni\MACHINE_LEARNING\python-for-ml\mini_projects\student_performance\data\processed

Figures directory:
C:\Users\LOQ\OneDrive\Documents\uni\MACHINE_LEARNING\python-for-ml\mini_projects\student_performance\reports\figures

Source directory:
C:\Users\LOQ\OneDrive\Documents\uni\MACHINE_LEARNING\python-for-ml\mini_projects\student_performance\src


## 3. Load the Data
This section loads the raw student_performance into a pandas DataFrame.
The raw data is kept unchanged. All cleaning and transformations will be performed later using copies of the original DataFrame.

In [8]:
print("Raw data path:", RAW_DATA_PATH)
print("File exists:", RAW_DATA_PATH.exists())

Raw data path: C:\Users\LOQ\OneDrive\Documents\uni\MACHINE_LEARNING\python-for-ml\mini_projects\student_performance\data\raw\student_performance.csv
File exists: True


In [9]:
raw_df = pd.read_csv(RAW_DATA_PATH)

In [10]:
print("Data loaded successfully.")
print("Shape:", raw_df.shape)

Data loaded successfully.
Shape: (201, 8)


In [11]:
raw_df.head()

,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score
0,STU001,Female,Software Engineering,11.2,84.1,7.4,7.8,9.6
1,STU002,Male,Software Engineering,5.8,96.9,7.9,7.8,5.7
2,STU003,Female,Software Engineering,13.0,81.1,9.8,9.1,8.0
3,STU004,Other,Software Engineering,13.8,87.7,8.8,8.1,9.6
4,STU005,Male,Computer Science,2.2,55.4,4.9,5.7,3.7


In [12]:
raw_df.columns.tolist()

['student_id',
 'gender',
 'major',
 'study_hours',
 'attendance_rate',
 'assignment_score',
 'midterm_score',
 'final_score']

In [13]:
type(raw_df)

pandas.DataFrame

## 4. Initial Data Inspection 
This section examines the dataset structure, column types, missing values, duplicate records, numerical summaries, and categorical values.
No data is modified in this section.

In [14]:
print("Number of rows:", raw_df.shape[0])
print("Number of columns:", raw_df.shape[1])

Number of rows: 201
Number of columns: 8


In [15]:
raw_df.head()


,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score
0,STU001,Female,Software Engineering,11.2,84.1,7.4,7.8,9.6
1,STU002,Male,Software Engineering,5.8,96.9,7.9,7.8,5.7
2,STU003,Female,Software Engineering,13.0,81.1,9.8,9.1,8.0
3,STU004,Other,Software Engineering,13.8,87.7,8.8,8.1,9.6
4,STU005,Male,Computer Science,2.2,55.4,4.9,5.7,3.7


In [16]:
raw_df.tail()

,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score
196,STU197,Other,Computer Science,12.6,89.1,8.8,8.6,9.2
197,STU198,Other,Computer Science,8.4,82.8,9.3,7.6,7.3
198,STU199,Other,Information Systems,10.0,86.4,8.8,8.5,8.2
199,STU200,Female,Information Systems,9.3,71.5,4.5,6.5,2.5
200,STU001,Female,Software Engineering,11.2,84.1,7.4,7.8,9.6


In [17]:
raw_df.dtypes

student_id              str
gender                  str
major                   str
study_hours         float64
attendance_rate     float64
assignment_score    float64
midterm_score       float64
final_score         float64
dtype: object

In [18]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   student_id        201 non-null    str    
 1   gender            201 non-null    str    
 2   major             201 non-null    str    
 3   study_hours       201 non-null    float64
 4   attendance_rate   201 non-null    float64
 5   assignment_score  200 non-null    float64
 6   midterm_score     201 non-null    float64
 7   final_score       201 non-null    float64
dtypes: float64(5), str(3)
memory usage: 12.7 KB


In [19]:
raw_df.describe().T

,count,mean,std,min,25%,50%,75%,max
study_hours,201.0,9.816418,3.635420,-3.0,7.4,9.8,12.1,21.7
attendance_rate,201.0,80.119403,11.800431,49.2,71.6,80.9,87.7,105.0
assignment_score,200.0,7.054000,1.713641,1.5,5.9,7.2,8.2,10.0
midterm_score,201.0,7.083085,1.701268,1.8,5.9,7.0,8.4,10.0
final_score,201.0,6.941294,1.997908,-1.0,5.5,7.0,8.6,10.0


In [20]:
raw_df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
student_id,201,200,STU001,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,201,4,Other,70,NaN,NaN,NaN,NaN,NaN,NaN,NaN
major,201,4,Software Engineering,86,NaN,NaN,NaN,NaN,NaN,NaN,NaN
study_hours,201.0,NaN,NaN,NaN,9.816418,3.63542,-3.0,7.4,9.8,12.1,21.7
attendance_rate,201.0,NaN,NaN,NaN,80.119403,11.800431,49.2,71.6,80.9,87.7,105.0
assignment_score,200.0,NaN,NaN,NaN,7.054,1.713641,1.5,5.9,7.2,8.2,10.0
midterm_score,201.0,NaN,NaN,NaN,7.083085,1.701268,1.8,5.9,7.0,8.4,10.0
final_score,201.0,NaN,NaN,NaN,6.941294,1.997908,-1.0,5.5,7.0,8.6,10.0


In [21]:
raw_df.isna().sum()

student_id          0
gender              0
major               0
study_hours         0
attendance_rate     0
assignment_score    1
midterm_score       0
final_score         0
dtype: int64

In [22]:
raw_df.duplicated().sum()

np.int64(1)

In [23]:
raw_df[
    raw_df.duplicated(keep=False)
]

,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score
0,STU001,Female,Software Engineering,11.2,84.1,7.4,7.8,9.6
200,STU001,Female,Software Engineering,11.2,84.1,7.4,7.8,9.6


In [24]:
raw_df["student_id"].duplicated().sum()

np.int64(1)

In [25]:
raw_df[
    raw_df["student_id"].duplicated(keep=False)
].sort_values("student_id")

,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score
0,STU001,Female,Software Engineering,11.2,84.1,7.4,7.8,9.6
200,STU001,Female,Software Engineering,11.2,84.1,7.4,7.8,9.6


In [26]:
raw_df["gender"].value_counts(dropna=False)
raw_df["major"].value_counts(dropna=False)

major
Software Engineering    86
Computer Science        58
Information Systems     56
 computer science        1
Name: count, dtype: int64

In [27]:
raw_df[
    raw_df["study_hours"] < 0
]

,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score
45,STU046,Male,Computer Science,-3.0,64.6,6.6,4.1,6.4


In [28]:
raw_df[
    ~raw_df["attendance_rate"].between(0, 100)
]

,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score
12,STU013,Male,Software Engineering,10.3,105.0,5.5,6.1,5.5


In [30]:
score_columns = [
    "assignment_score",
    "midterm_score",
    "final_score",
]

for column in score_columns:
    invalid_rows = raw_df[
        raw_df[column].notna()
        & ~raw_df[column].between(0, 10)
    ]

    print(f"{column}: {len(invalid_rows)} invalid rows")
    display(invalid_rows)

assignment_score: 0 invalid rows


,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score


midterm_score: 0 invalid rows


,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score


final_score: 1 invalid rows


,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score
30,STU031,Other,Software Engineering,18.6,62.4,7.4,7.0,-1.0


### Initial Inspection Findings

- The dataset contains 201 rows and 8 columns.
- One exact duplicate row was detected.
- One missing value was found in `assignment_score`.
- An invalid negative value was found in `study_hours`.
- An attendance value greater than 100 was detected.
- An invalid negative value was found in `final_score`.
- Some categorical values contain extra whitespace or inconsistent capitalization.